In [1]:
import dspy
from typing import Literal
from dotenv import load_dotenv
import os

load_dotenv()

ANTHROPIC_KEY = os.getenv("ANTHROPIC_KEY")
OPENAI_KEY = os.getenv("OPENAI_KEY")

In [2]:
student_model = dspy.LM("anthropic/claude-haiku-4-5-20251001", api_key=ANTHROPIC_KEY)

#### Contract clause risk assessor 

You receive a clause from a legal contract. Classify the risk_level (high, medium, low) and clause_type (liability, termination, payment, intellectual_property, confidentiality, indemnification).

In [3]:
clauses = [
    {"clause": "The vendor shall not be liable for any indirect, incidental, or consequential damages, regardless of cause, even if advised of the possibility of such damages", "risk_level": "high", "clause_type": "liability"},
    {"clause": "Either party may terminate this agreement with 30 days written notice without cause", "risk_level": "low", "clause_type": "termination"},
    {"clause": "Payment is due within 90 days of invoice receipt, with no penalties for late payment", "risk_level": "medium", "clause_type": "payment"},
    {"clause": "All intellectual property created during the engagement shall be owned exclusively by the vendor, including derivative works", "risk_level": "high", "clause_type": "intellectual_property"},
    {"clause": "Confidential information shall be protected for a period of 2 years following termination of this agreement", "risk_level": "low", "clause_type": "confidentiality"},
    {"clause": "The client shall indemnify and hold harmless the vendor against any and all claims, damages, and expenses without limitation", "risk_level": "high", "clause_type": "indemnification"},
    {"clause": "The vendor may terminate this agreement immediately if the client fails to make payment within 15 days of the due date", "risk_level": "medium", "clause_type": "termination"},
    {"clause": "All payments are non-refundable under any circumstances, including early termination or failure to deliver", "risk_level": "high", "clause_type": "payment"},
    {"clause": "The client retains ownership of all pre-existing intellectual property provided to the vendor during the engagement", "risk_level": "low", "clause_type": "intellectual_property"},
    {"clause": "The vendor agrees to maintain reasonable security measures to protect client data during the term of the agreement", "risk_level": "low", "clause_type": "confidentiality"},
    {"clause": "The vendor's total liability under this agreement shall not exceed the fees paid in the 12 months preceding the claim", "risk_level": "medium", "clause_type": "liability"},
    {"clause": "Upon termination, the client shall pay for all work completed plus a 25% early termination fee", "risk_level": "medium", "clause_type": "termination"},
    {"clause": "Each party shall indemnify the other only for claims arising from its own negligence or willful misconduct", "risk_level": "low", "clause_type": "indemnification"},
    {"clause": "The vendor retains an irrevocable, royalty-free license to use any client data for training machine learning models", "risk_level": "high", "clause_type": "intellectual_property"},
    {"clause": "Confidentiality obligations shall survive indefinitely and cover all information exchanged verbally or in writing", "risk_level": "medium", "clause_type": "confidentiality"},
    {"clause": "The client shall be liable for any third-party claims arising from the use of deliverables, even if caused by vendor defects", "risk_level": "high", "clause_type": "liability"},
    {"clause": "The vendor may subcontract any portion of the work without prior client approval or notification", "risk_level": "high", "clause_type": "liability"},
    {"clause": "Disputes shall be resolved through binding arbitration in the vendor's home jurisdiction at the client's expense", "risk_level": "high", "clause_type": "liability"},
    {"clause": "The client may terminate for cause with 14 days written notice and a reasonable opportunity to cure", "risk_level": "low", "clause_type": "termination"},
    {"clause": "Late payments shall incur interest at 1.5% per month compounding until paid in full", "risk_level": "medium", "clause_type": "payment"},
    {"clause": "Both parties agree to mutual non-disclosure of proprietary information for 5 years post-termination", "risk_level": "low", "clause_type": "confidentiality"},
    {"clause": "The vendor shall own all custom code, configurations, and documentation produced under this agreement, with no license granted back to the client", "risk_level": "high", "clause_type": "intellectual_property"},
]

In [4]:
##Create test/train sets

full_data = [dspy.Example(clause).with_inputs("clause") for clause in clauses]
train_data = full_data[0:10]
test_data = full_data[10:]

#Create signature
class ContractClause(dspy.Signature):
    """ You are an expert Contract Clause Assessor. Given a clause I want you to classify each
        clause into 2 buckets - risk_level and clause_type
    """

    clause:str = dspy.InputField(desc="A brief description of a clause in a contract")
    risk_level:Literal["high", "medium", "low"] = dspy.OutputField(desc="""How unfavorable 
    the clause is for the client. 
    High means the client is exposed or giving up significant rights.""")
    clause_type:Literal["liability", "termination", "payment", "intellectual_property",\
    "confidentiality", "indemnification"] = dspy.OutputField(desc="The legal domain the clause falls under.")

#Create an evaluation function - 100% match
def evaluate_clause_classification(true, pred, trace=None):

    return (true.risk_level.lower() == pred.risk_level.lower()) and (
        true.clause_type.lower() == pred.clause_type.lower())


#Create an evaluator object
def evaluate_module(module, path_to_store_logs):
    contract_clause_evaluator = dspy.Evaluate(
        devset=test_data,
        metric=evaluate_clause_classification,
        display_progress=True,
        save_as_json=path_to_store_logs)

    return contract_clause_evaluator(module)


In [5]:
##Unoptimized predict module

dspy.configure(lm=dspy.LM("anthropic/claude-haiku-4-5-20251001", api_key=ANTHROPIC_KEY))
unoptimized_clause_assessor = dspy.ChainOfThought(ContractClause)

print(f"Accuracy in unoptimized assessor - {evaluate_module(unoptimized_clause_assessor, "./exploration_evaluation_results/unoptimized_mipro.json")}")

#Quite accurate already good and very high

Average Metric: 11.00 / 12 (91.7%): 100%|██████████████████| 12/12 [00:00<00:00, 25.12it/s]

2026/03/19 16:15:24 INFO dspy.evaluate.evaluate: Average Metric: 11 / 12 (91.7%)



Accuracy in unoptimized assessor - EvaluationResult(score=91.67, results=<list of 12 results>)


In [6]:
dspy.inspect_history(n=1)





[2026-03-19T16:15:24.921710]

System message:

Your input fields are:
1. `clause` (str): A brief description of a clause in a contract
Your output fields are:
1. `reasoning` (str): 
2. `risk_level` (Literal['high', 'medium', 'low']): How unfavorable 
    the clause is for the client. 
    High means the client is exposed or giving up significant rights.
3. `clause_type` (Literal['liability', 'termination', 'payment', 'intellectual_property', 'confidentiality', 'indemnification']): The legal domain the clause falls under.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## clause ## ]]
{clause}

[[ ## reasoning ## ]]
{reasoning}

[[ ## risk_level ## ]]
{risk_level}        # note: the value you produce must exactly match (no extra characters) one of: high; medium; low

[[ ## clause_type ## ]]
{clause_type}        # note: the value you produce must exactly match (no extra characters) one of: liability; termination; payment; intellect

## MIPROV2 

Iteration 1 -> Same student/task and prompt model

In [7]:
support_model = dspy.LM("anthropic/claude-haiku-4-5-20251001", api_key=ANTHROPIC_KEY)

miprov_iter1 = dspy.MIPROv2(
    metric=evaluate_clause_classification,
    prompt_model=support_model,
    task_model=support_model,
    max_bootstrapped_demos=7,
    max_labeled_demos=10,
    auto="medium",
    init_temperature=0.1
)

miprov1_optimized = miprov_iter1.compile(
    student=unoptimized_clause_assessor,
    trainset=train_data,
)

2026/03/19 16:15:24 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING MEDIUM AUTO RUN SETTINGS:
num_trials: 18
minibatch: False
num_fewshot_candidates: 12
num_instruct_candidates: 6
valset size: 8

2026/03/19 16:15:24 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/03/19 16:15:24 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/03/19 16:15:24 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=12 sets of demonstrations...


Bootstrapping set 1/12
Bootstrapping set 2/12
Bootstrapping set 3/12


100%|████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 17.92it/s]


Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 4/12


100%|████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 20.70it/s]


...

2026/03/19 16:15:36 INFO dspy.evaluate.evaluate: Average Metric: 5 / 8 (62.5%)
2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 62.5 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 11'].
2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [75.0, 62.5, 62.5, 62.5, 62.5, 75.0, 62.5, 50.0, 75.0, 62.5, 75.0, 75.0, 62.5, 75.0, 62.5, 62.5, 62.5]
2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 75.0
2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 18 / 18 =====



Average Metric: 5.00 / 8 (62.5%): 100%|█████████████████████| 8/8 [00:00<00:00, 527.40it/s]

2026/03/19 16:15:36 INFO dspy.evaluate.evaluate: Average Metric: 5 / 8 (62.5%)
2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 62.5 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 5'].
2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [75.0, 62.5, 62.5, 62.5, 62.5, 75.0, 62.5, 50.0, 75.0, 62.5, 75.0, 75.0, 62.5, 75.0, 62.5, 62.5, 62.5, 62.5]
2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 75.0
2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/19 16:15:36 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 19 / 18 =====



Average Metric: 4.00 / 8 (50.0%): 100%|██████████████████████| 8/8 [00:04<00:00,  1.73it/s]

2026/03/19 16:15:41 INFO dspy.evaluate.evaluate: Average Metric: 4 / 8 (50.0%)
2026/03/19 16:15:41 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 50.0 with parameters ['Predictor 0: Instruction 5', 'Predictor 0: Few-Shot Set 0'].
2026/03/19 16:15:41 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [75.0, 62.5, 62.5, 62.5, 62.5, 75.0, 62.5, 50.0, 75.0, 62.5, 75.0, 75.0, 62.5, 75.0, 62.5, 62.5, 62.5, 62.5, 50.0]
2026/03/19 16:15:41 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 75.0
2026/03/19 16:15:41 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/19 16:15:41 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 75.0!


In [8]:

evaluate_module(miprov1_optimized, "./exploration_evaluation_results/miprov1.json")

Average Metric: 11.00 / 12 (91.7%): 100%|█████████████████| 12/12 [00:00<00:00, 657.74it/s]

2026/03/19 16:15:41 INFO dspy.evaluate.evaluate: Average Metric: 11 / 12 (91.7%)


EvaluationResult(score=91.67, results=<list of 12 results>)

In [9]:
dspy.inspect_history(n=1) #didnt CHanges the system prompt. but worse accuracy





[2026-03-19T16:15:41.399789]

System message:

Your input fields are:
1. `clause` (str): A brief description of a clause in a contract
Your output fields are:
1. `reasoning` (str): 
2. `risk_level` (Literal['high', 'medium', 'low']): How unfavorable 
    the clause is for the client. 
    High means the client is exposed or giving up significant rights.
3. `clause_type` (Literal['liability', 'termination', 'payment', 'intellectual_property', 'confidentiality', 'indemnification']): The legal domain the clause falls under.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## clause ## ]]
{clause}

[[ ## reasoning ## ]]
{reasoning}

[[ ## risk_level ## ]]
{risk_level}        # note: the value you produce must exactly match (no extra characters) one of: high; medium; low

[[ ## clause_type ## ]]
{clause_type}        # note: the value you produce must exactly match (no extra characters) one of: liability; termination; payment; intellect

Iteration 2 - 
Student model is claude-haiku-4-5-20251001 

Prompt model is claude-opus-4-6

In [10]:
prompt_model = dspy.LM("anthropic/claude-opus-4-6", api_key=ANTHROPIC_KEY)

miprov_iter2 = dspy.MIPROv2(
    metric=evaluate_clause_classification,
    prompt_model=prompt_model, 
    task_model=support_model,
    max_bootstrapped_demos=7,
    max_labeled_demos=10,
    auto="medium",
    init_temperature=0.1
)

miprov2_optimized = miprov_iter2.compile(
    student=unoptimized_clause_assessor,
    trainset=train_data,
)

2026/03/19 16:15:41 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING MEDIUM AUTO RUN SETTINGS:
num_trials: 18
minibatch: False
num_fewshot_candidates: 12
num_instruct_candidates: 6
valset size: 8

2026/03/19 16:15:41 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/03/19 16:15:41 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/03/19 16:15:41 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=12 sets of demonstrations...


Bootstrapping set 1/12
Bootstrapping set 2/12
Bootstrapping set 3/12


100%|████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 17.24it/s]


Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 4/12


100%|████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 29.53it/s]


...

2026/03/19 16:15:51 INFO dspy.evaluate.evaluate: Average Metric: 6 / 8 (75.0%)
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 75.0 with parameters ['Predictor 0: Instruction 3', 'Predictor 0: Few-Shot Set 7'].
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [75.0, 62.5, 75.0, 62.5, 87.5, 75.0, 75.0, 75.0, 75.0, 87.5, 75.0, 87.5, 87.5, 87.5, 87.5, 87.5, 75.0]
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 87.5
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 18 / 18 =====



Average Metric: 5.00 / 8 (62.5%): 100%|█████████████████████| 8/8 [00:00<00:00, 664.92it/s]

2026/03/19 16:15:51 INFO dspy.evaluate.evaluate: Average Metric: 5 / 8 (62.5%)
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 62.5 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 4'].
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [75.0, 62.5, 75.0, 62.5, 87.5, 75.0, 75.0, 75.0, 75.0, 87.5, 75.0, 87.5, 87.5, 87.5, 87.5, 87.5, 75.0, 62.5]
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 87.5
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 19 / 18 =====



Average Metric: 7.00 / 8 (87.5%): 100%|█████████████████████| 8/8 [00:00<00:00, 471.84it/s]

2026/03/19 16:15:51 INFO dspy.evaluate.evaluate: Average Metric: 7 / 8 (87.5%)
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 87.5 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 11'].
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [75.0, 62.5, 75.0, 62.5, 87.5, 75.0, 75.0, 75.0, 75.0, 87.5, 75.0, 87.5, 87.5, 87.5, 87.5, 87.5, 75.0, 62.5, 87.5]
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 87.5
2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/19 16:15:51 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 87.5!


In [11]:
evaluate_module(miprov2_optimized, "./exploration_evaluation_results/miprov2.json")

#Performs worse than benchmark

Average Metric: 8.00 / 12 (66.7%): 100%|██████████████████| 12/12 [00:00<00:00, 178.28it/s]

2026/03/19 16:15:51 INFO dspy.evaluate.evaluate: Average Metric: 8 / 12 (66.7%)


EvaluationResult(score=66.67, results=<list of 12 results>)

In [12]:
dspy.inspect_history(n=1) #changed system prompt





[2026-03-19T16:15:51.796948]

System message:

Your input fields are:
1. `clause` (str): A brief description of a clause in a contract
Your output fields are:
1. `reasoning` (str): 
2. `risk_level` (Literal['high', 'medium', 'low']): How unfavorable 
    the clause is for the client. 
    High means the client is exposed or giving up significant rights.
3. `clause_type` (Literal['liability', 'termination', 'payment', 'intellectual_property', 'confidentiality', 'indemnification']): The legal domain the clause falls under.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## clause ## ]]
{clause}

[[ ## reasoning ## ]]
{reasoning}

[[ ## risk_level ## ]]
{risk_level}        # note: the value you produce must exactly match (no extra characters) one of: high; medium; low

[[ ## clause_type ## ]]
{clause_type}        # note: the value you produce must exactly match (no extra characters) one of: liability; termination; payment; intellect